<a href="https://colab.research.google.com/github/eyacharfeddine/alpr-using-domain-adaptation-/blob/main/integration_and_inference.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Install latest compatible versions
!pip install --upgrade ultralytics onnx

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.6/17.6 MB 115.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 118.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 97.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 60.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 41.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 19.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 93.8 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu

In [ ]:
from ultralytics import YOLO
import os
import shutil # For moving the file

# --- PATHS ---
# 1. Path to your existing YOLOv8 .pt model (VERIFY THIS)
YOLO_MODEL_PATH_PT = "/content/drive/MyDrive/pguard/domain_adaptation_process1/visible_to_thermal_semisupervised/weights/best.pt"

# 2. Desired FULL PATH for the exported ONNX model (includes the directory and filename)
YOLO_ONNX_EXPORT_FULL_PATH = "/content/drive/MyDrive/pguard/detection_model.onnx"

# 3. Image size used for ONNX export
YOLO_IMG_SIZE_FOR_EXPORT = 640

print(f"--- Starting YOLO ONNX Export (Corrected for 'export_path' error) ---")
print(f"Source PyTorch model: {YOLO_MODEL_PATH_PT}")
print(f"Target ONNX model path: {YOLO_ONNX_EXPORT_FULL_PATH}")

# Ensure the source .pt file exists
if not os.path.exists(YOLO_MODEL_PATH_PT):
    print(f"ERROR: Source YOLO .pt file not found at '{YOLO_MODEL_PATH_PT}'. Please check the path.")
else:
    try:
        # Ensure the target directory exists. If not, create it.
        target_onnx_directory = os.path.dirname(YOLO_ONNX_EXPORT_FULL_PATH)
        if not os.path.exists(target_onnx_directory):
            print(f"Target directory '{target_onnx_directory}' does not exist. Creating it...")
            os.makedirs(target_onnx_directory, exist_ok=True)

        print("Loading YOLO model from .pt file...")
        yolo_model = YOLO(YOLO_MODEL_PATH_PT)
        print(f"YOLO model loaded successfully. Preparing for ONNX export with image size: {YOLO_IMG_SIZE_FOR_EXPORT}x{YOLO_IMG_SIZE_FOR_EXPORT}")

        # Perform the export WITHOUT `export_path`
        print(f"Calling yolo_model.export() without 'export_path' argument...")
        yolo_model.export(
            format="onnx",
            imgsz=YOLO_IMG_SIZE_FOR_EXPORT,
            half=False,
            device="cpu",
            simplify=True
        )
        print("YOLO model.export() call completed.")

        # Determine the expected default name of the exported ONNX file
        base_pt_name = os.path.splitext(os.path.basename(YOLO_MODEL_PATH_PT))[0] # e.g., "best"
        default_onnx_filename = base_pt_name + ".onnx" # e.g., "best.onnx"

        # Possible locations where ultralytics might have saved the file
        location_in_cwd = os.path.join(os.getcwd(), default_onnx_filename) # e.g., /content/best.onnx
        location_in_original_dir = os.path.join(os.path.dirname(YOLO_MODEL_PATH_PT), default_onnx_filename)

        exported_file_found_at = None
        if os.path.exists(location_in_cwd):
            exported_file_found_at = location_in_cwd
            print(f"ONNX file found in current working directory: '{location_in_cwd}'")
        elif os.path.exists(location_in_original_dir):
            exported_file_found_at = location_in_original_dir
            print(f"ONNX file found in original model's directory: '{location_in_original_dir}'")

        if exported_file_found_at:
            # Move the found ONNX file to the final desired path, overwriting if it exists
            shutil.move(exported_file_found_at, YOLO_ONNX_EXPORT_FULL_PATH)
            print(f"Successfully exported and moved ONNX model to: '{YOLO_ONNX_EXPORT_FULL_PATH}'")
        else:
            print(f"ERROR: Exported ONNX file ('{default_onnx_filename}') not found at common default locations:")
            print(f"  - Current working directory: '{location_in_cwd}'")
            print(f"  - Original model's directory: '{location_in_original_dir}'")
            print(f"Please check the console for Ultralytics messages about where the file was saved, and manually move it to '{YOLO_ONNX_EXPORT_FULL_PATH}' if the export was otherwise successful.")

    except Exception as e:
        print(f"ERROR during YOLO export process: {e}")
        import traceback
        traceback.print_exc()

print("--- YOLO ONNX Export Script Finished ---")

--- Starting YOLO ONNX Export (Corrected for 'export_path' error) ---
Source PyTorch model: /content/drive/MyDrive/pguard/domain_adaptation_process1/visible_to_thermal_semisupervised/weights/best.pt
Target ONNX model path: /content/drive/MyDrive/pguard/detection_model.onnx
Loading YOLO model from .pt file...
YOLO model loaded successfully. Preparing for ONNX export with image size: 640x640
Calling yolo_model.export() without 'export_path' argument...
Ultralytics 8.3.145 🚀 Python-3.11.12 torch-2.6.0+cu124 CPU (Intel Xeon 2.20GHz)
💡 ProTip: Export to OpenVINO format for best performance on Intel CPUs. Learn more at https://docs.ultralytics.com/integrations/openvino/
Model summary (fused): 72 layers, 11,125,971 parameters, 0 gradients, 28.4 GFLOPs

PyTorch: starting from '/content/drive/MyDrive/pguard/domain_adaptation_process1/visible_to_thermal_semisupervised/weights/best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 5, 8400) (21.5 MB)
requirements: Ultralytics requ

In [ ]:
!git clone https://github.com/ayumiymk/aster.pytorch.git

Cloning into 'aster.pytorch'...
remote: Enumerating objects: 86, done.
remote: Counting objects: 100% (37/37), done.
remote: Compressing objects: 100% (17/17), done.
remote: Total 86 (delta 21), reused 20 (delta 20), pack-reused 49 (from 1)
Receiving objects: 100% (86/86), 115.12 KiB | 2.26 MiB/s, done.
Resolving deltas: 100% (22/22), done.


In [ ]:
%%writefile /content/aster.pytorch/lib/loss/sequenceCrossEntropyLoss.py
from __future__ import absolute_import

import torch
from torch import nn
import torch.nn.functional as F
import logging

logger = logging.getLogger(__name__)


def to_contiguous(tensor):
    if tensor.is_contiguous():
        return tensor
    else:
        return tensor.contiguous()

def _assert_no_grad(variable):
    assert not variable.requires_grad, \
        "nn criterions don't compute the gradient w.r.t. targets - please " \
        "mark these variables as not requiring gradients"

class SequenceCrossEntropyLoss(nn.Module):
    def __init__(self,
                 weight=None,
                 size_average=True, # Conceptually, for reduction
                 ignore_index=-100, # Standard ignore_index, though not directly used for masking here
                 sequence_normalize=False, # If True, loss is per-token (sum of losses / num active tokens)
                 sample_normalize=True,  # If True (and sequence_normalize=False), loss is per-sample (sum of losses / batch_size)
                 label_smoothing=0.0):
        super(SequenceCrossEntropyLoss, self).__init__()
        self.weight = weight # Not used in ASTER's typical setup
        self.ignore_index = ignore_index # Not used for explicit masking here
        self.sequence_normalize = sequence_normalize
        self.sample_normalize = sample_normalize
        self.label_smoothing = label_smoothing

        if sequence_normalize and sample_normalize:
            logger.warning("Both sequence_normalize and sample_normalize are True. "
                           "sequence_normalize will take precedence in the current logic.")


    def forward(self, input_logits, target_labels, sequence_lengths):
        # input_logits: (batch_size, max_pred_len, num_classes)
        # target_labels: (batch_size, max_gt_len_padded)
        # sequence_lengths: (batch_size,) - true length of each sequence in target_labels

        _assert_no_grad(target_labels)

        batch_size = target_labels.size(0)
        if batch_size == 0: # Handle empty batch
            return torch.tensor(0.0, device=input_logits.device, requires_grad=True)

        max_true_len_in_batch = 0
        if sequence_lengths.numel() > 0: # Ensure sequence_lengths is not empty
            max_true_len_in_batch = torch.max(sequence_lengths).item()
            if not isinstance(max_true_len_in_batch, int): # Ensure it's an int
                 max_true_len_in_batch = int(max_true_len_in_batch)
        if max_true_len_in_batch == 0 and batch_size > 0 : # All sequences have zero length
             # logger.warning("All sequences in batch have zero true length. Returning 0 loss.")
             return torch.tensor(0.0, device=input_logits.device, requires_grad=True)


        max_padded_len_targets = target_labels.size(1)
        pred_len_logits = input_logits.size(1)

        # Determine the effective length for loss calculation:
        # minimum of max true length, predicted length, and padded target length
        effective_len = min(max_true_len_in_batch, pred_len_logits, max_padded_len_targets)

        if effective_len <= 0: # If no valid elements to compute loss over
            return torch.tensor(0.0, device=input_logits.device, requires_grad=True)

        # Create mask based on actual sequence_lengths, up to effective_len
        mask = torch.zeros(batch_size, effective_len, device=input_logits.device, dtype=torch.bool)
        for i in range(batch_size):
            true_len_i = sequence_lengths[i].item()
            # Mask elements up to the minimum of true_len_i and effective_len
            mask[i, :min(true_len_i, effective_len)] = True
        mask = mask.type_as(input_logits) # Convert boolean mask to float (0.0 or 1.0)

        # Truncate tensors to effective_len
        input_logits_trunc = input_logits[:, :effective_len, :]
        target_labels_trunc = target_labels[:, :effective_len]
        # mask is already shaped for effective_len

        # Flatten tensors for cross-entropy
        input_logits_flat = to_contiguous(input_logits_trunc).view(-1, input_logits_trunc.size(2))
        target_labels_flat = to_contiguous(target_labels_trunc).view(-1)
        mask_flat = to_contiguous(mask).view(-1)

        num_classes = input_logits_flat.size(1)
        if num_classes == 0:
             logger.error("Number of classes is 0 in SequenceCrossEntropyLoss. Vocab issue? Returning 0 loss.")
             return torch.tensor(0.0, device=input_logits.device, requires_grad=True)

        log_probs = F.log_softmax(input_logits_flat, dim=1)

        if self.label_smoothing > 0.0:
            # Ensure num_classes > 1 for label smoothing denominator
            denominator_ls = (num_classes - 1) if num_classes > 1 else 1.0
            if denominator_ls == 0: denominator_ls = 1.0 # Safeguard

            one_hot_targets = torch.zeros_like(log_probs).scatter_(1, target_labels_flat.long().unsqueeze(1), 1)
            smooth_targets_dist = one_hot_targets * (1 - self.label_smoothing) + \
                                  (1 - one_hot_targets) * self.label_smoothing / denominator_ls
            loss_elements = - (log_probs * smooth_targets_dist).sum(dim=1)
        else:
            loss_elements = -log_probs.gather(1, target_labels_flat.long().unsqueeze(1)).squeeze(1)

        masked_loss_elements = loss_elements * mask_flat
        total_loss = torch.sum(masked_loss_elements)
        num_active_elements = torch.sum(mask_flat)

        if num_active_elements.item() == 0:
            return torch.tensor(0.0, device=input_logits.device, requires_grad=True)

        # Normalization
        if self.sequence_normalize:
            final_loss = total_loss / num_active_elements
        elif self.sample_normalize:
            final_loss = total_loss / batch_size
        else:
            final_loss = total_loss

        if torch.isnan(final_loss) or torch.isinf(final_loss):
            logger.error(f"NaN/Inf detected in final_loss! Logits min/max/mean: "
                         f"{input_logits.min().item():.2f}/{input_logits.max().item():.2f}/{input_logits.mean().item():.2f}. "
                         f"Total loss: {total_loss.item()}, Active elements: {num_active_elements.item()}, BS: {batch_size}")
            final_loss = torch.tensor(0.0, device=input_logits.device, requires_grad=True) # Prevent training crash

        return final_loss

Overwriting /content/aster.pytorch/lib/loss/sequenceCrossEntropyLoss.py


In [ ]:
%%writefile /content/aster.pytorch/lib/models/model_builder.py
from __future__ import absolute_import

import torch
from torch import nn
from torch.nn import functional as F
from torch.nn import init
from torch.cuda.amp import autocast # For mixed precision
import logging

from . import create
from .attention_recognition_head import AttentionRecognitionHead
from .tps_spatial_transformer import TPSSpatialTransformer
from .stn_head import STNHead
from ..loss.sequenceCrossEntropyLoss import SequenceCrossEntropyLoss

logger = logging.getLogger(__name__)

class GradientReversalFunction(torch.autograd.Function):
    @staticmethod
    @autocast(enabled=False) # GRL should operate in full precision
    def forward(ctx, x, lambda_val):
        ctx.lambda_val = lambda_val
        return x.view_as(x)

    @staticmethod
    @autocast(enabled=False) # GRL should operate in full precision
    def backward(ctx, grad_output):
        return grad_output.neg() * ctx.lambda_val, None

def gradient_reversal(x, lambda_val=1.0):
    return GradientReversalFunction.apply(x, lambda_val)

class DomainDiscriminator(nn.Module):
    def __init__(self, in_planes, hidden_dim=512, dropout_rate=0.3):
        super(DomainDiscriminator, self).__init__()
        self.layers = nn.Sequential(
            nn.Linear(in_planes, hidden_dim),
            nn.BatchNorm1d(hidden_dim), # Consider LayerNorm if BN causes issues with small batches/GRL
            nn.LeakyReLU(0.2, inplace=True),
            nn.Dropout(dropout_rate),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.BatchNorm1d(hidden_dim // 2), # Consider LayerNorm
            nn.LeakyReLU(0.2, inplace=True),
            nn.Dropout(dropout_rate),
            nn.Linear(hidden_dim // 2, 1)
        )
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                init.kaiming_normal_(m.weight, a=0.2, mode='fan_out', nonlinearity='leaky_relu')
                if m.bias is not None:
                    init.constant_(m.bias, 0)
            elif isinstance(m, nn.BatchNorm1d):
                init.constant_(m.weight, 1)
                init.constant_(m.bias, 0)

    def forward(self, x):
        return self.layers(x)


class ModelBuilder(nn.Module):
    def __init__(self, arch, rec_num_classes, sDim, attDim, max_len_labels, eos,
                 STN_ON=False, domain_adaptation=False,
                 tps_inputsize=(32, 64), tps_outputsize=(32, 100),
                 num_control_points=20, tps_margins=(0.05,0.05),
                 stn_activation='none', with_lstm=True, n_group=1,
                 label_smoothing=0.0, da_feature_norm=False): # Added da_feature_norm
        super(ModelBuilder, self).__init__()
        self.arch = arch
        self.rec_num_classes = rec_num_classes
        self.sDim = sDim
        self.attDim = attDim
        self.max_len_labels = max_len_labels
        self.eos = eos
        self.STN_ON = STN_ON
        self.domain_adaptation = domain_adaptation
        self.with_lstm = with_lstm
        self.da_feature_norm = da_feature_norm # Store this

        if self.STN_ON:
            self.tps_inputsize = tuple(tps_inputsize)
            self.tps_outputsize = tuple(tps_outputsize)
            self.num_control_points = num_control_points
            self.tps_margins = tuple(tps_margins)
            self.stn_activation = stn_activation
            self.tps = TPSSpatialTransformer(
                output_image_size=self.tps_outputsize,
                num_control_points=self.num_control_points,
                margins=self.tps_margins)
            self.stn_head = STNHead(
                in_planes=3, # Assumes 3-channel input for STN head
                num_ctrlpoints=self.num_control_points,
                activation=self.stn_activation)

        self.encoder = create(self.arch, with_lstm=self.with_lstm, n_group=n_group if hasattr(self, 'n_group') else 1)
        self.decoder = AttentionRecognitionHead(
            num_classes=rec_num_classes,
            in_planes=self.encoder.out_planes,
            sDim=sDim,
            attDim=attDim,
            max_len_labels=max_len_labels)
        self.rec_crit = SequenceCrossEntropyLoss(label_smoothing=label_smoothing)
        self.grl_lambda = 1.0 # Default value, will be updated by trainer

        if self.domain_adaptation:
            self.domain_discriminator = DomainDiscriminator(in_planes=self.encoder.out_planes)
            self.domain_crit = nn.BCEWithLogitsLoss()
            if self.da_feature_norm:
                # Ensure the LayerNorm is compatible with the feature dimension
                self.da_feat_layernorm = nn.LayerNorm(self.encoder.out_planes)
                logger.info("LayerNorm enabled for features used in DA components.")


    def _apply_stn(self, x):
        # Ensure input to STN head is 3-channel if that's what it expects
        if tuple(x.shape[2:]) != tuple(self.tps_inputsize):
            stn_input_for_stnhead_resized = F.interpolate(x, self.tps_inputsize, mode='bilinear', align_corners=True)
        else:
            stn_input_for_stnhead_resized = x

        stn_input_for_stnhead_c = stn_input_for_stnhead_resized.size(1)
        if stn_input_for_stnhead_c != 3:
            # logger.warning(f"STN head expects 3 input channels, got {stn_input_for_stnhead_c}. Will attempt to use first 3 or replicate if 1.")
            if stn_input_for_stnhead_c == 1:
                stn_input_for_stnhead = stn_input_for_stnhead_resized.repeat(1,3,1,1)
            else: # more than 1 but not 3, take first 3 or error
                 stn_input_for_stnhead = stn_input_for_stnhead_resized[:, :3, :, :]
        else:
            stn_input_for_stnhead = stn_input_for_stnhead_resized


        _, ctrl_points = self.stn_head(stn_input_for_stnhead)
        # TPS module itself might handle 1-channel or 3-channel input for warping,
        # typically warps the original input 'x'
        x_rectified, _ = self.tps(x, ctrl_points)
        return x_rectified, ctrl_points

    def forward(self, input_dict):
        return_dict = {'losses': {}, 'output': {}}

        images = input_dict['images']
        # is_source: True for source, False for target (unlabeled OR SSL labeled)
        is_source = input_dict.get('is_source', True)
        rec_targets = input_dict.get('rec_targets', None)
        rec_lengths = input_dict.get('rec_lengths', None)
        current_grl_lambda = self.grl_lambda


        images_for_features = images
        if self.STN_ON:
            images_rectified, ctrl_points = self._apply_stn(images) # Pass original images to _apply_stn
            if torch.isnan(images_rectified).any() or torch.isinf(images_rectified).any():
                domain_log = "Source" if is_source else ("SSL Target" if rec_targets is not None else "Unlabeled Target")
                logger.warning(f"NaN/Inf in image tensor after STN. Clamping. Domain: {domain_log}")
                images_rectified = torch.nan_to_num(images_rectified, nan=0.0, posinf=1.0, neginf=-1.0)
            if not self.training:
                return_dict['output']['ctrl_points'] = ctrl_points
                return_dict['output']['rectified_images'] = images_rectified
            images_for_features = images_rectified

        encoder_feats_for_rec = self.encoder(images_for_features)
        encoder_feats_for_rec = encoder_feats_for_rec.contiguous()


        # --- Recognition Loss Logic (Source & SSL Target) ---
        if self.training:
            # Compute recognition loss if actual labels are provided (for source or SSL target)
            if rec_targets is not None and rec_lengths is not None and rec_lengths.sum() > 0 :
                rec_pred_logits = self.decoder((encoder_feats_for_rec, rec_targets, rec_lengths))
                loss_rec_val = self.rec_crit(rec_pred_logits, rec_targets, rec_lengths)

                if torch.isnan(loss_rec_val) or torch.isinf(loss_rec_val):
                     domain_str = "Source" if is_source else "SSL Target"
                     logger.error(f"NaN/Inf detected in LossRec ({domain_str})! Logits min/max/mean: "
                                  f"{rec_pred_logits.min().item():.2f}/{rec_pred_logits.max().item():.2f}/{rec_pred_logits.mean().item():.2f}. Setting loss to 0.")
                     loss_rec_val = torch.tensor(0.0, device=loss_rec_val.device, requires_grad=True) # Allow backward for other losses
                return_dict['losses']['loss_rec'] = loss_rec_val.unsqueeze(0)
            # else: # No valid labels for rec loss (e.g. unlabeled target batch)
            #    pass # loss_rec will not be in return_dict['losses'] for this path

        else: # Evaluation phase
            if rec_targets is not None and rec_lengths is not None and rec_lengths.sum() > 0:
                eval_rec_pred_logits = self.decoder((encoder_feats_for_rec, rec_targets, rec_lengths))
                eval_loss_rec = self.rec_crit(eval_rec_pred_logits, rec_targets, rec_lengths)
                return_dict['losses']['loss_rec_eval'] = eval_loss_rec.unsqueeze(0)

            beam_width = input_dict.get('beam_width', 5)
            rec_pred_ids, rec_pred_scores = self.decoder.beam_search(encoder_feats_for_rec, beam_width=beam_width, eos=self.eos)
            return_dict['output']['pred_rec'] = rec_pred_ids
            return_dict['output']['pred_rec_score'] = rec_pred_scores


        # --- Domain Adaptation Logic ---
        if self.domain_adaptation:
            # Prepare features for domain classifier and MMD
            if encoder_feats_for_rec.dim() == 3: # (batch, seq_len, feat_dim)
                feat_for_domain_raw = encoder_feats_for_rec.mean(dim=1)
            elif encoder_feats_for_rec.dim() == 4: # (batch, channels, H, W)
                feat_for_domain_raw = F.adaptive_avg_pool2d(encoder_feats_for_rec, (1,1)).view(encoder_feats_for_rec.size(0), -1)
            else:
                raise ValueError(f"Unsupported encoder_feats_for_rec shape for DA: {encoder_feats_for_rec.shape}")

            feat_for_domain = feat_for_domain_raw
            if self.da_feature_norm and hasattr(self, 'da_feat_layernorm'):
                feat_for_domain = self.da_feat_layernorm(feat_for_domain_raw)
            # Detach features for MMD loss (MMD is usually calculated on detached features)
            # For adversarial loss, GRL handles the gradient reversal, so features passed to discriminator should not be detached here.
            return_dict['output']['domain_feat'] = feat_for_domain.detach()


            if self.training: # Adversarial loss only during training
                feat_grl = gradient_reversal(feat_for_domain, current_grl_lambda)
                domain_pred_logits = self.domain_discriminator(feat_grl)

                # Adversarial target labels for the *generator* (feature extractor):
                # - If input is source (is_source=True), generator wants discriminator to predict it as target (label 1).
                # - If input is target (is_source=False), generator wants discriminator to predict it as source (label 0).
                adv_target_labels = torch.ones_like(domain_pred_logits, device=images.device) if is_source \
                               else torch.zeros_like(domain_pred_logits, device=images.device)

                loss_adv = self.domain_crit(domain_pred_logits, adv_target_labels)
                if torch.isnan(loss_adv) or torch.isinf(loss_adv):
                    logger.error(f"NaN/Inf in Adv Loss. Clamping to 0. is_source: {is_source}")
                    loss_adv = torch.tensor(0.0, device=loss_adv.device, requires_grad=True)
                return_dict['losses']['loss_adv'] = loss_adv.unsqueeze(0)

        return return_dict

Overwriting /content/aster.pytorch/lib/models/model_builder.py


In [ ]:
%%writefile /content/aster.pytorch/config.py
from __future__ import absolute_import
from __future__ import division
from __future__ import print_function
import sys
sys.path.append('./') # This is usually for when scripts are run from within the repo root.

import os
import os.path as osp
import math
import argparse # Make sure argparse is imported
import logging
from datetime import datetime
import torch

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S'
)
logger = logging.getLogger(__name__)

parser = argparse.ArgumentParser(description="ASTER Domain Adaptation with SSL, MMD, and Annealed Weights")

# Data
parser.add_argument('--synthetic_train_data_dir', nargs='+', type=str, metavar='PATH',
                    default=['/content/drive/MyDrive/pguard/train/lmdb'])
parser.add_argument('--val_data_dir', type=str, metavar='PATH',
                    default='/content/drive/MyDrive/pguard/val_original/lmdb')
parser.add_argument('--test_data_dir', type=str, metavar='PATH',
                    default='/content/drive/MyDrive/pguard//Cars/test_lmdb') # Note: double slash here, might be intentional or a typo
parser.add_argument('--target_data_dir', nargs='+', type=str, metavar='PATH', # For UNLABELED target
                    default=['/content/drive/MyDrive/pguard/Cars/train_lmdb'])
parser.add_argument('--target_val_data_dir', type=str, metavar='PATH',
                    default='/content/drive/MyDrive/pguard/Cars/val_lmdb')
# --- SSL Specific Data ---
parser.add_argument('--labeled_target_data_dir', nargs='+', type=str, metavar='PATH', default=None,
                    help="Path to LMDB for LABELED target domain data (for SSL).")
parser.add_argument('--num_labeled_target', type=int, default=100,
                    help="Number of LABELED target samples to use for SSL (per epoch if smaller than dataset).")


parser.add_argument('-b', '--batch_size', type=int, default=32)
parser.add_argument('--ssl_batch_size', type=int, default=8, help="Batch size for the SSL (labeled target) dataloader.")
parser.add_argument('-j', '--workers', type=int, default=2)
parser.add_argument('--height', type=int, default=32)
parser.add_argument('--width', type=int, default=100)
parser.add_argument('--keep_ratio', action='store_true', default=False)
parser.add_argument('--voc_type', type=str, default='LICENSE_PLATE')
parser.add_argument('--num_train', type=int, default=3200)
parser.add_argument('--num_target', type=int, default=949)
parser.add_argument('--num_val', type=int, default=400)
parser.add_argument('--num_target_val', type=int, default=204)
parser.add_argument('--num_test', type=int, default=400)
parser.add_argument('--aug', action='store_true', default=True) # Note: If using in notebook, this might need explicit True/False if default behavior doesn't activate
parser.add_argument('--source_rgb', action='store_true', default=True, help="Is source dataset RGB?")
parser.add_argument('--target_rgb', action='store_true', default=False, help="Is target dataset RGB? (False for grayscale thermal)")


# Model
parser.add_argument('-a', '--arch', type=str, default='ResNet_ASTER')
parser.add_argument('--max_len', type=int, default=11)
parser.add_argument('--STN_ON', action='store_true', default=False)
parser.add_argument('--tps_inputsize', nargs='+', type=int, default=[32, 64])
parser.add_argument('--tps_outputsize', nargs='+', type=int, default=[32, 100])
parser.add_argument('--tps_margins', nargs='+', type=float, default=[0.05, 0.05])
parser.add_argument('--stn_activation', type=str, default='none')
parser.add_argument('--num_control_points', type=int, default=20)
parser.add_argument('--with_lstm', action='store_true', default=True) # Note: Same as --aug, default True.
parser.add_argument('--decoder_sdim', type=int, default=512)
parser.add_argument('--attDim', type=int, default=512)
parser.add_argument('--label_smoothing', type=float, default=0.1)

# Optimizer
parser.add_argument('--lr', type=float, default=1e-4)
parser.add_argument('--lr_encoder_early_ratio', type=float, default=0.1)
parser.add_argument('--lr_discriminator', type=float, default=5e-5)
parser.add_argument('--lr_adapt_phase_ratio', type=float, default=0.5)
parser.add_argument('--weight_decay', type=float, default=1e-5)
parser.add_argument('--grad_clip', type=float, default=5.0)
parser.add_argument('--scheduler_patience_phase1', type=int, default=5)
parser.add_argument('--scheduler_patience_phase2', type=int, default=7)
parser.add_argument('--scheduler_factor', type=float, default=0.5)


# Training
parser.add_argument('--resume', type=str, default='/content/drive/MyDrive/pguard/recognitionmodel1/best.pth.tar')
parser.add_argument('--evaluate', action='store_true', default=False)
parser.add_argument('--epochs', type=int, default=80)
parser.add_argument('--start_epoch', type=int, default=0)
parser.add_argument('--seed', type=int, default=1111)
parser.add_argument('--print_freq', type=int, default=50)
parser.add_argument('--eval_freq', type=int, default=1)
parser.add_argument('--no_cuda', action='store_true', default=False)
parser.add_argument('--use_amp', action='store_true', default=True) # Default True.
parser.add_argument('--patience_early_stop', type=int, default=20)
parser.add_argument('--warmup_epochs', type=int, default=3)
parser.add_argument('--stabilize_source_epochs', type=int, default=10)


# Domain Adaptation
parser.add_argument('--domain_adaptation', action='store_true', default=True) # Default True.
parser.add_argument('--target_specific_aug', action='store_true', default=True) # Default True.
parser.add_argument('--grl_alpha', type=float, default=10.0)
parser.add_argument('--grl_final_high', type=float, default=0.1)
parser.add_argument('--grl_warmup_epochs', type=int, default=10)
parser.add_argument('--grl_initial_high', type=float, default=0.01)
parser.add_argument('--lw_rec', type=float, default=1.0)
parser.add_argument('--lw_adv_base', type=float, default=0.1)
parser.add_argument('--anneal_adv_epochs', nargs=2, type=int, default=[5, 25])
parser.add_argument('--anneal_adv_start_weight', type=float, default=0.005)
parser.add_argument('--lw_ssl_rec_tgt_base', type=float, default=0.75)
parser.add_argument('--anneal_ssl_rec_tgt_epochs', nargs=2, type=int, default=[1, 20])
parser.add_argument('--anneal_ssl_rec_tgt_start_weight', type=float, default=0.05)
parser.add_argument('--lw_mmd_base', type=float, default=0.05)
parser.add_argument('--anneal_mmd_epochs', nargs=2, type=int, default=[10, 30])
parser.add_argument('--anneal_mmd_start_weight', type=float, default=0.001)
parser.add_argument('--mmd_kernel_mul', type=float, default=2.0)
parser.add_argument('--mmd_kernel_num', type=int, default=5)
parser.add_argument('--da_feature_norm', action='store_true', default=True) # Default True.

# Testing
parser.add_argument('--evaluation_metric', type=str, default='accuracy')
parser.add_argument('--evaluate_with_lexicon', action='store_true', default=False)
parser.add_argument('--beam_width', type=int, default=5)

# Misc
parser.add_argument('--run_on_remote', action='store_true', default=False)
parser.add_argument('--logs_dir_name', type=str, default='logs_da_thermal_rev1')
parser.add_argument('--real_logs_dir', type=str, metavar='PATH', default='/content/drive/MyDrive/pguard/da_training_logs')
parser.add_argument('--vis_dir_name', type=str, default='visualizations')
parser.add_argument('--source_model_save_dir', type=str, default='/content/drive/MyDrive/pguard/recognitionmodel1')
parser.add_argument('--da_model_save_dir', type=str, default='/content/drive/MyDrive/pguard/Cars/domainadaptation_thermal_rev1')
parser.add_argument('--debug', action='store_true', default=False)

_parsed_args = None

def get_args(sys_args=None): # Default sys_args to None
    global _parsed_args
    if _parsed_args is None:
        # --- MODIFICATION HERE ---
        # Check if sys_args are from Jupyter/Colab or if it's empty (when imported)
        if sys_args is None or (isinstance(sys_args, list) and any(arg.startswith('-f') and '.json' in arg for arg in sys_args)):
            print("INFO (config.py get_args): No explicit CLI args or Jupyter/Colab args detected. Parsing with empty list to get defaults.")
            _parsed_args = parser.parse_args([]) # Parse with NO command line args to get defaults
        else:
            # This is the original behavior, for when config.py is run as a script with actual arguments
            _parsed_args = parser.parse_args(sys_args)
        # --- END OF MODIFICATION ---

        _parsed_args.cuda = not _parsed_args.no_cuda and torch.cuda.is_available()

        # --- Robust logging setup (check for attributes before using) ---
        log_file_handler_path = None
        current_logs_dir_set = False
        if hasattr(_parsed_args, 'evaluate'):
            if hasattr(_parsed_args, 'real_logs_dir') and hasattr(_parsed_args, 'logs_dir_name'):
                base_log_dir = _parsed_args.real_logs_dir
                log_subdir_name = _parsed_args.logs_dir_name

                if not _parsed_args.evaluate:
                    _parsed_args.current_logs_dir = osp.join(base_log_dir, log_subdir_name, datetime.now().strftime('%Y-%m-%d_%H-%M-%S'))
                    log_file_handler_path = osp.join(_parsed_args.current_logs_dir, 'training.log')
                    current_logs_dir_set = True
                else:
                    eval_log_parent_dir = osp.join(base_log_dir, log_subdir_name)
                    _parsed_args.current_logs_dir = osp.join(eval_log_parent_dir, "evaluation_run_" + datetime.now().strftime('%Y-%m-%d_%H-%M-%S'))
                    log_file_handler_path = osp.join(_parsed_args.current_logs_dir, 'evaluation.log')
                    current_logs_dir_set = True

                if current_logs_dir_set:
                    os.makedirs(_parsed_args.current_logs_dir, exist_ok=True)
            else:
                logger.warning("real_logs_dir or logs_dir_name not defined in args. Log file setup skipped.")
        else:
            logger.warning("'evaluate' attribute not found in args. Log file setup might be incomplete.")

        root_logger_instance = logging.getLogger() # Use a different variable name to avoid conflict with logger module
        if hasattr(_parsed_args, 'real_logs_dir'):
            for handler in root_logger_instance.handlers[:]:
                if isinstance(handler, logging.FileHandler):
                    if hasattr(handler, 'baseFilename') and _parsed_args.real_logs_dir in handler.baseFilename:
                        root_logger_instance.removeHandler(handler)
                        handler.close()

        if not any(isinstance(h, logging.StreamHandler) for h in root_logger_instance.handlers):
            ch = logging.StreamHandler(sys.stdout)
            ch.setFormatter(logging.Formatter('%(asctime)s - %(name)s - %(levelname)s - %(message)s', datefmt='%Y-%m-%d %H:%M-%S'))
            root_logger_instance.addHandler(ch)

        if log_file_handler_path:
            file_handler = logging.FileHandler(log_file_handler_path, mode='a')
            file_handler.setFormatter(logging.Formatter('%(asctime)s - %(name)s - %(levelname)s - %(message)s', datefmt='%Y-%m-%d %H:%M-%S'))
            root_logger_instance.addHandler(file_handler)
            logger.info("Logger configured in config.py. Log file: %s", log_file_handler_path) # Use the module-level logger here
        else:
            logger.info("Logger configured in config.py without a file handler (log_file_handler_path not set).")

        root_logger_instance.setLevel(logging.INFO)

    return _parsed_args


Overwriting /content/aster.pytorch/config.py


In [ ]:
%%writefile /content/aster.pytorch/lib/loss/sequenceCrossEntropyLoss.py
from __future__ import absolute_import

import torch
from torch import nn
import torch.nn.functional as F
import logging

logger = logging.getLogger(__name__)


def to_contiguous(tensor):
    if tensor.is_contiguous():
        return tensor
    else:
        return tensor.contiguous()

def _assert_no_grad(variable):
    assert not variable.requires_grad, \
        "nn criterions don't compute the gradient w.r.t. targets - please " \
        "mark these variables as not requiring gradients"

class SequenceCrossEntropyLoss(nn.Module):
    def __init__(self,
                 weight=None,
                 size_average=True, # Conceptually, for reduction
                 ignore_index=-100, # Standard ignore_index, though not directly used for masking here
                 sequence_normalize=False, # If True, loss is per-token (sum of losses / num active tokens)
                 sample_normalize=True,  # If True (and sequence_normalize=False), loss is per-sample (sum of losses / batch_size)
                 label_smoothing=0.0):
        super(SequenceCrossEntropyLoss, self).__init__()
        self.weight = weight # Not used in ASTER's typical setup
        self.ignore_index = ignore_index # Not used for explicit masking here
        self.sequence_normalize = sequence_normalize
        self.sample_normalize = sample_normalize
        self.label_smoothing = label_smoothing

        if sequence_normalize and sample_normalize:
            logger.warning("Both sequence_normalize and sample_normalize are True. "
                           "sequence_normalize will take precedence in the current logic.")


    def forward(self, input_logits, target_labels, sequence_lengths):
        # input_logits: (batch_size, max_pred_len, num_classes)
        # target_labels: (batch_size, max_gt_len_padded)
        # sequence_lengths: (batch_size,) - true length of each sequence in target_labels

        _assert_no_grad(target_labels)

        batch_size = target_labels.size(0)
        if batch_size == 0: # Handle empty batch
            return torch.tensor(0.0, device=input_logits.device, requires_grad=True)

        max_true_len_in_batch = 0
        if sequence_lengths.numel() > 0: # Ensure sequence_lengths is not empty
            max_true_len_in_batch = torch.max(sequence_lengths).item()
            if not isinstance(max_true_len_in_batch, int): # Ensure it's an int
                 max_true_len_in_batch = int(max_true_len_in_batch)
        if max_true_len_in_batch == 0 and batch_size > 0 : # All sequences have zero length
             # logger.warning("All sequences in batch have zero true length. Returning 0 loss.")
             return torch.tensor(0.0, device=input_logits.device, requires_grad=True)


        max_padded_len_targets = target_labels.size(1)
        pred_len_logits = input_logits.size(1)

        # Determine the effective length for loss calculation:
        # minimum of max true length, predicted length, and padded target length
        effective_len = min(max_true_len_in_batch, pred_len_logits, max_padded_len_targets)

        if effective_len <= 0: # If no valid elements to compute loss over
            return torch.tensor(0.0, device=input_logits.device, requires_grad=True)

        # Create mask based on actual sequence_lengths, up to effective_len
        mask = torch.zeros(batch_size, effective_len, device=input_logits.device, dtype=torch.bool)
        for i in range(batch_size):
            true_len_i = sequence_lengths[i].item()
            # Mask elements up to the minimum of true_len_i and effective_len
            mask[i, :min(true_len_i, effective_len)] = True
        mask = mask.type_as(input_logits) # Convert boolean mask to float (0.0 or 1.0)

        # Truncate tensors to effective_len
        input_logits_trunc = input_logits[:, :effective_len, :]
        target_labels_trunc = target_labels[:, :effective_len]
        # mask is already shaped for effective_len

        # Flatten tensors for cross-entropy
        input_logits_flat = to_contiguous(input_logits_trunc).view(-1, input_logits_trunc.size(2))
        target_labels_flat = to_contiguous(target_labels_trunc).view(-1)
        mask_flat = to_contiguous(mask).view(-1)

        num_classes = input_logits_flat.size(1)
        if num_classes == 0:
             logger.error("Number of classes is 0 in SequenceCrossEntropyLoss. Vocab issue? Returning 0 loss.")
             return torch.tensor(0.0, device=input_logits.device, requires_grad=True)

        log_probs = F.log_softmax(input_logits_flat, dim=1)

        if self.label_smoothing > 0.0:
            # Ensure num_classes > 1 for label smoothing denominator
            denominator_ls = (num_classes - 1) if num_classes > 1 else 1.0
            if denominator_ls == 0: denominator_ls = 1.0 # Safeguard

            one_hot_targets = torch.zeros_like(log_probs).scatter_(1, target_labels_flat.long().unsqueeze(1), 1)
            smooth_targets_dist = one_hot_targets * (1 - self.label_smoothing) + \
                                  (1 - one_hot_targets) * self.label_smoothing / denominator_ls
            loss_elements = - (log_probs * smooth_targets_dist).sum(dim=1)
        else:
            loss_elements = -log_probs.gather(1, target_labels_flat.long().unsqueeze(1)).squeeze(1)

        masked_loss_elements = loss_elements * mask_flat
        total_loss = torch.sum(masked_loss_elements)
        num_active_elements = torch.sum(mask_flat)

        if num_active_elements.item() == 0:
            return torch.tensor(0.0, device=input_logits.device, requires_grad=True)

        # Normalization
        if self.sequence_normalize:
            final_loss = total_loss / num_active_elements
        elif self.sample_normalize:
            final_loss = total_loss / batch_size
        else:
            final_loss = total_loss

        if torch.isnan(final_loss) or torch.isinf(final_loss):
            logger.error(f"NaN/Inf detected in final_loss! Logits min/max/mean: "
                         f"{input_logits.min().item():.2f}/{input_logits.max().item():.2f}/{input_logits.mean().item():.2f}. "
                         f"Total loss: {total_loss.item()}, Active elements: {num_active_elements.item()}, BS: {batch_size}")
            final_loss = torch.tensor(0.0, device=input_logits.device, requires_grad=True) # Prevent training crash

        return final_loss

Overwriting /content/aster.pytorch/lib/loss/sequenceCrossEntropyLoss.py


In [ ]:
import os
import sys
import shutil # For file operations if needed later
import torch
import torch.nn as nn # Needed by ASTER's ModelBuilder indirectly
import torch.nn.functional as F # Needed by ASTER's ModelBuilder indirectly
from torch.nn import init # Needed by ASTER's ModelBuilder indirectly

In [ ]:
%%writefile /content/aster.pytorch/lib/utils/visualization_utils.py
from __future__ import absolute_import

from PIL import Image
import os
import numpy as np
from collections import OrderedDict
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from io import BytesIO
from multiprocessing import Pool
import math
import sys

import torch
from torch.nn import functional as F

from . import to_torch, to_numpy
from ..evaluation_metrics.metrics import get_str_list

def recognition_vis(images, preds, targets, scores, dataset, vis_dir):
    images = images.permute(0, 2, 3, 1)
    images = to_numpy(images)
    images = (images * 0.5 + 0.5) * 255
    pred_list, targ_list = get_str_list(preds, targets, dataset)
    for id, (image, pred, target, score) in enumerate(zip(images, pred_list, targ_list, scores)):
        if pred.lower() == target.lower():
            flag = 'right'
        else:
            flag = 'error'
        file_name = '{:}_{:}_{:}_{:}_{:.3f}.jpg'.format(flag, id, pred, target, score)
        file_path = os.path.join(vis_dir, file_name)
        image = Image.fromarray(np.uint8(image))
        image.save(file_path)

# Save to disk sub process
def _save_plot_pool(vis_image, save_file_path):
    vis_image = Image.fromarray(np.uint8(vis_image))
    vis_image.save(save_file_path)

def stn_vis(raw_images, rectified_images, ctrl_points, preds, targets, real_scores, pred_scores, dataset, vis_dir):
    """
    raw_images: images without rectification
    rectified_images: rectified images with stn
    ctrl_points: predicted ctrl points
    preds: predicted label sequences
    targets: target label sequences
    real_scores: scores of recognition model
    pred_scores: predicted scores by the score branch
    dataset: xxx
    vis_dir: xxx
    """
    if raw_images.ndimension() == 3:
        raw_images = raw_images.unsqueeze(0)
        rectified_images = rectified_images.unsqueeze(0)
    batch_size, _, raw_height, raw_width = raw_images.size()

    # Translate the coordinates of ctrl points to image size
    ctrl_points = to_numpy(ctrl_points)
    ctrl_points[:, :, 0] = ctrl_points[:, :, 0] * (raw_width - 1)
    ctrl_points[:, :, 1] = ctrl_points[:, :, 1] * (raw_height - 1)
    ctrl_points = ctrl_points.astype(np.int)

    # Convert tensors to PIL images
    raw_images = raw_images.permute(0, 2, 3, 1)
    raw_images = to_numpy(raw_images)
    raw_images = (raw_images * 0.5 + 0.5) * 255
    rectified_images = rectified_images.permute(0, 2, 3, 1)
    rectified_images = to_numpy(rectified_images)
    rectified_images = (rectified_images * 0.5 + 0.5) * 255

    # Draw images on canvas
    vis_images = []
    num_sub_plot = 2
    raw_images = raw_images.astype(np.uint8)
    rectified_images = rectified_images.astype(np.uint8)
    for i in range(batch_size):
        fig = plt.figure()
        ax = [fig.add_subplot(num_sub_plot, 1, i + 1) for i in range(num_sub_plot)]
        for a in ax:
            a.set_xticklabels([])
            a.set_yticklabels([])
            a.axis('off')
        ax[0].imshow(raw_images[i])
        ax[0].scatter(ctrl_points[i, :, 0], ctrl_points[i, :, 1], marker='+', s=5)
        ax[1].imshow(rectified_images[i])
        buffer_ = BytesIO()
        plt.savefig(buffer_, format='png', bbox_inches='tight', pad_inches=0)
        plt.close()
        buffer_.seek(0)
        dataPIL = Image.open(buffer_)
        data = np.asarray(dataPIL).astype(np.uint8)
        buffer_.close()
        vis_images.append(data)

    # Save to disk
    if vis_dir is None:
        return vis_images
    else:
        pred_list, targ_list = get_str_list(preds, targets, dataset)
        file_path_list = []
        for id, (image, pred, target, real_score) in enumerate(zip(vis_images, pred_list, targ_list, real_scores)):
            if pred.lower() == target.lower():
                flag = 'right'
            else:
                flag = 'error'
            if pred_scores is None:
                file_name = '{:}_{:}_{:}_{:}_{:.3f}.png'.format(flag, id, pred, target, real_score)
            else:
                file_name = '{:}_{:}_{:}_{:}_{:.3f}_{:.3f}.png'.format(flag, id, pred, target, real_score, pred_scores[id])
            file_path = os.path.join(vis_dir, file_name)
            file_path_list.append(file_path)

        with Pool(os.cpu_count()) as pool:
            pool.starmap(_save_plot_pool, zip(vis_images, file_path_list))

Overwriting /content/aster.pytorch/lib/utils/visualization_utils.py


In [ ]:
%%writefile /content/aster.pytorch/lib/utils/labelmaps.py
from __future__ import absolute_import
import string
import torch  # Keep for type checking and potential tensor operations
import numpy as np

print("--- Writing updated labelmaps.py (v5 - ndim fix and id2char fix) ---")

# Define to_torch and to_numpy locally for self-containment if `from . import ...` fails
# These are helper functions; if your ASTER library defines them centrally in utils/__init__.py, that's also fine.
def to_torch(x):
    if isinstance(x, np.ndarray):
        return torch.from_numpy(x)
    elif not isinstance(x, torch.Tensor): # Basic fallback for other types if needed
        # This part depends on what other types might be passed.
        # For safety, if it's not ndarray or tensor, and torch conversion is needed,
        # one might need to handle it specifically or raise an error.
        # However, for labels2strs, input will primarily be tensor or ndarray.
        try:
            return torch.tensor(x) # Attempt generic conversion
        except Exception as e:
            print(f"Warning (to_torch): Could not convert type {type(x)} to torch.Tensor: {e}")
            return x # Return as is if conversion fails or is not obvious
    return x

def to_numpy(x): # Ensures output is a NumPy array
    if isinstance(x, torch.Tensor):
        return x.cpu().detach().numpy() if hasattr(x, 'cpu') else x.detach().numpy()
    elif not isinstance(x, np.ndarray): # If not a tensor and not already numpy
        return np.array(x) # Convert other types (like lists) to numpy
    return x


def get_vocabulary(voc_type, EOS='EOS', PADDING='PADDING', UNKNOWN='UNKNOWN'):
    '''
    voc_type: str: one of 'LOWERCASE', 'ALLCASES', 'ALLCASES_SYMBOLS', 'LICENSE_PLATE'
    '''
    voc = None
    # This list of types must match the if/elif conditions below
    supported_types = ['LOWERCASE', 'ALLCASES', 'ALLCASES_SYMBOLS', 'LICENSE_PLATE']

    if voc_type == 'LOWERCASE':
        voc = list(string.digits + string.ascii_lowercase)
    elif voc_type == 'ALLCASES':
        voc = list(string.digits + string.ascii_letters)
    elif voc_type == 'ALLCASES_SYMBOLS':
        voc = list(string.printable[:-6])
    elif voc_type == 'LICENSE_PLATE':
        voc = list('0123456789ABCDEFGHIJKLMNOPQRSTUVWXYZ#-')
    else:
        # This error message now correctly lists all supported types including LICENSE_PLATE
        raise KeyError(f"voc_type '{voc_type}' is invalid. Must be one of {supported_types}")

    # Ensure special tokens are part of the vocabulary list if not already present
    # (they usually are by design of how voc is constructed, but this is a safeguard)
    if EOS not in voc: voc.append(EOS)
    if PADDING not in voc: voc.append(PADDING)
    if UNKNOWN not in voc: voc.append(UNKNOWN)
    return voc

def char2id(voc_list): # Expects the list from get_vocabulary
    return dict(zip(voc_list, range(len(voc_list))))

def id2char(voc_list): # Expects the list from get_vocabulary
    # Corrected implementation
    return dict(zip(range(len(voc_list)), voc_list))

def labels2strs(labels_input, id2char_map_arg, char2id_map_arg):
    # labels_input can be PyTorch tensor (from training/eval) or NumPy array (from ONNX output)

    # Convert to NumPy array at the beginning for consistent handling
    np_labels = to_numpy(labels_input)

    # Use .ndim for NumPy arrays
    if np_labels.ndim == 1:
        np_labels = np.expand_dims(np_labels, 0) # Reshape to (1, seq_len) if single sequence

    # Ensure it's 2D (batch_size, seq_len)
    if np_labels.ndim != 2:
        print(f"WARNING (labels2strs): Input labels have unexpected shape {np_labels.shape}. Expected 2D array (batch, seq_len).")
        # Attempt to handle or return error strings
        if np_labels.shape[0] > 0:
             return [f"SHAPE_ERROR_{np_labels.shape}" for _ in range(np_labels.shape[0])] # One error string per batch item
        return [f"SHAPE_ERROR_{np_labels.shape}"] # Fallback for completely uninterpretable shape


    strings_list = []
    # Get IDs of special tokens from the provided char2id_map for efficient lookup
    eos_id_val = char2id_map_arg.get('EOS')
    # padding_id_val = char2id_map_arg.get('PADDING') # Not strictly needed if we just skip EOS and beyond
    # unknown_id_val = char2id_map_arg.get('UNKNOWN') # Not strictly needed for string construction

    for label_sequence in np_labels: # Iterate over each sequence in the batch
        current_char_list = []
        for char_code_float_or_int in label_sequence: # char_code can be float from some ONNX runtimes
            char_code = int(char_code_float_or_int) # Convert to int for dict key

            if eos_id_val is not None and char_code == eos_id_val:
                break # Stop decoding at the first End-Of-Sequence token

            character = id2char_map_arg.get(char_code)

            # Append character if it's valid and not a special non-printing token
            # (assuming PADDING and UNKNOWN should not be part of the final string)
            if character and character not in ['PADDING', 'UNKNOWN', 'EOS']: # Check against string names
                current_char_list.append(character)
        strings_list.append("".join(current_char_list))

    return strings_list



Overwriting /content/aster.pytorch/lib/utils/labelmaps.py


In [ ]:
%%writefile /content/aster.pytorch/lib/models/attention_recognition_head.py
from __future__ import absolute_import

import sys
import torch
from torch import nn
from torch.nn import functional as F
from torch.nn import init

# --- Start of AttentionUnit and DecoderUnit definitions ---
# (Using the definitions you provided in your last `%%writefile` for attention_recognition_head.py,
#  as they seem to be the ones from your ayumiymk/aster.pytorch clone)

class AttentionUnit(nn.Module):
    def __init__(self, sDim, xDim, attDim):
        super(AttentionUnit, self).__init__()
        self.sDim = sDim; self.xDim = xDim; self.attDim = attDim
        self.sEmbed = nn.Linear(sDim, attDim)
        self.xEmbed = nn.Linear(xDim, attDim)
        self.wEmbed = nn.Linear(attDim, 1)
        for m in [self.sEmbed, self.xEmbed, self.wEmbed]: # Basic init
            init.xavier_uniform_(m.weight)
            if m.bias is not None: init.constant_(m.bias, 0)

    def forward(self, encoder_features, decoder_hidden_prev):
        batch_size, T_encoder, _ = encoder_features.size()
        sPrev_squeezed = decoder_hidden_prev.squeeze(0)
        sProj_expanded = self.sEmbed(sPrev_squeezed).unsqueeze(1).expand(batch_size, T_encoder, self.attDim)
        encoder_features_flat = encoder_features.contiguous().view(-1, self.xDim)
        xProj_flat = self.xEmbed(encoder_features_flat)
        xProj = xProj_flat.view(batch_size, T_encoder, self.attDim)
        sum_tanh_flat = torch.tanh(sProj_expanded + xProj).view(-1, self.attDim)
        vProj_flat = self.wEmbed(sum_tanh_flat)
        vProj = vProj_flat.view(batch_size, T_encoder)
        return F.softmax(vProj, dim=1)

class DecoderUnit(nn.Module):
    def __init__(self, sDim, xDim, yDim, attDim):
        super(DecoderUnit, self).__init__()
        self.sDim, self.xDim, self.yDim, self.attDim, self.emdDim = sDim, xDim, yDim, attDim, attDim
        self.attention_unit = AttentionUnit(sDim, xDim, attDim)
        self.tgt_embedding = nn.Embedding(yDim + 1, self.emdDim)
        self.gru = nn.GRU(input_size=xDim + self.emdDim, hidden_size=sDim, batch_first=True)
        self.fc = nn.Linear(sDim, yDim)
        init.xavier_uniform_(self.tgt_embedding.weight)
        init.xavier_uniform_(self.fc.weight)
        if self.fc.bias is not None: init.constant_(self.fc.bias, 0)

    def forward(self, encoder_features, decoder_hidden_prev, prev_char_ids):
        attention_weights = self.attention_unit(encoder_features, decoder_hidden_prev)
        context_vector = torch.bmm(attention_weights.unsqueeze(1), encoder_features).squeeze(1)
        embedded_prev_char = self.tgt_embedding(prev_char_ids.long())
        gru_input_one_step = torch.cat([embedded_prev_char, context_vector], 1).unsqueeze(1)
        output_gru, next_hidden_state = self.gru(gru_input_one_step, decoder_hidden_prev)
        output_logits = self.fc(output_gru.squeeze(1))
        return output_logits, next_hidden_state
# --- End of AttentionUnit and DecoderUnit definitions ---

class AttentionRecognitionHead(nn.Module):
    def __init__(self, num_classes, in_planes, sDim, attDim, max_len_labels):
        super(AttentionRecognitionHead, self).__init__()
        self.num_classes = num_classes
        self.in_planes = in_planes; self.sDim = sDim; self.attDim = attDim
        self.max_len_labels = max_len_labels
        self.decoder = DecoderUnit(sDim=sDim, xDim=in_planes, yDim=num_classes, attDim=attDim)

    def forward(self, x_input_tuple): # TRAINING FORWARD
        encoder_features, targets, target_lengths = x_input_tuple
        batch_size = encoder_features.size(0)
        decoder_hidden_state = torch.zeros(1, batch_size, self.sDim, device=encoder_features.device)
        outputs = []
        max_len_in_batch = max(target_lengths) if isinstance(target_lengths, list) else target_lengths.max().item()
        for i in range(max_len_in_batch):
            prev_char_ids = targets[:, i-1] if i > 0 else torch.full((batch_size,), self.num_classes, dtype=torch.long, device=encoder_features.device)
            current_logits, decoder_hidden_state = self.decoder(encoder_features, decoder_hidden_state, prev_char_ids)
            outputs.append(current_logits)
        return torch.stack(outputs, dim=1) if outputs else torch.empty(batch_size, 0, self.num_classes, device=encoder_features.device)

    # ====================================================================
    # CORRECTED SAMPLE METHOD FOR INFERENCE & ONNX EXPORT
    # ====================================================================
    def sample(self, encoder_features):
        # `encoder_features` is the direct tensor output from the encoder.
        # DO NOT unpack it here like `encoder_features, _, _ = encoder_features`

        batch_size = encoder_features.size(0)
        decoder_hidden_state = torch.zeros(1, batch_size, self.sDim, device=encoder_features.device)

        predicted_ids_list = []
        predicted_scores_list = []

        current_input_token_ids = torch.full((batch_size,), self.num_classes, dtype=torch.long, device=encoder_features.device) # BOS token

        for _ in range(self.max_len_labels):
            step_logits, decoder_hidden_state = self.decoder(
                encoder_features,
                decoder_hidden_state,
                current_input_token_ids
            )
            step_probs = F.softmax(step_logits, dim=1)
            scores, predicted_next_token_ids = step_probs.max(1) # Greedy selection

            predicted_ids_list.append(predicted_next_token_ids.unsqueeze(1))
            predicted_scores_list.append(scores.unsqueeze(1))
            current_input_token_ids = predicted_next_token_ids

        final_ids = torch.cat(predicted_ids_list, dim=1)
        final_scores = torch.cat(predicted_scores_list, dim=1)
        return final_ids, final_scores
    # ====================================================================
    # END OF CORRECTED SAMPLE METHOD
    # ====================================================================

    # BEAM SEARCH (Provided by you - complex for ONNX, .sample() is preferred for ONNXWrapper)
    def beam_search(self, encoder_features_x, beam_width, eos_id):
        print("WARNING (ARH beam_search): Beam search called. For ONNX, .sample() is more robust.")
        # Using the simplified beam search structure for ONNX traceability if it has to be used
        # Note: Your wrapper is set to use .sample(), so this should not be hit during ONNX export with current wrapper.

        def _inflate(tensor, times, dim):
            repeat_dims = [1] * tensor.dim(); repeat_dims[dim] = times
            return tensor.repeat(*repeat_dims)

        batch_size, seq_len_enc, feat_dim_enc = encoder_features_x.size()
        inflated_encoder_feats = encoder_features_x.unsqueeze(1).repeat(1, beam_width, 1, 1).view(batch_size * beam_width, seq_len_enc, feat_dim_enc)
        state = torch.zeros(1, batch_size * beam_width, self.sDim, device=encoder_features_x.device)
        pos_index = (torch.arange(batch_size, device=encoder_features_x.device) * beam_width).long().view(-1, 1)
        sequence_scores = torch.full((batch_size * beam_width, 1), -float('Inf'), device=encoder_features_x.device)
        initial_beam_indices = torch.arange(0, batch_size * beam_width, beam_width, device=encoder_features_x.device)
        sequence_scores.index_fill_(0, initial_beam_indices, 0.0)
        y_prev_char_ids = torch.full((batch_size * beam_width,), self.num_classes, dtype=torch.long, device=encoder_features_x.device)

        stored_scores_list, stored_predecessors_list, stored_emitted_symbols_list = [], [], []

        for _i_step in range(self.max_len_labels):
            output_logits, state = self.decoder(inflated_encoder_feats, state, y_prev_char_ids)
            log_softmax_output = F.log_softmax(output_logits, dim=1)
            current_total_scores = sequence_scores + log_softmax_output
            scores_reshaped = current_total_scores.view(batch_size, -1)
            top_k_scores, top_k_candidates_indices = scores_reshaped.topk(beam_width, dim=1)
            y_prev_char_ids = (top_k_candidates_indices % self.num_classes).view(batch_size * beam_width)
            sequence_scores = top_k_scores.view(batch_size * beam_width, 1)
            predecessors_global_beam_idx = ((top_k_candidates_indices // self.num_classes) + pos_index).view(batch_size * beam_width)
            state = state.index_select(1, predecessors_global_beam_idx)
            stored_scores_list.append(sequence_scores.clone())
            eos_mask = y_prev_char_ids.eq(eos_id)
            if eos_mask.any(): sequence_scores.masked_fill_(eos_mask.unsqueeze(1), -float('Inf'))
            stored_predecessors_list.append(predecessors_global_beam_idx.unsqueeze(1))
            stored_emitted_symbols_list.append(y_prev_char_ids)

        # Simplified Backtracking - This is a placeholder and might not give optimal beam search results
        final_sequences = []
        last_step_scores = stored_scores_list[-1].view(batch_size, beam_width)
        _, best_final_beam_indices = last_step_scores.topk(1, dim=1) # Take top-1 beam from each batch
        for b_idx in range(batch_size):
            best_beam_global_idx = b_idx * beam_width + best_final_beam_indices[b_idx, 0]
            current_sequence_chars = [eos_id] * self.max_len_labels # Initialize with EOS
            temp_beam_idx_for_trace = best_beam_global_idx
            for t_step in range(self.max_len_labels - 1, -1, -1):
                char_symbol = stored_emitted_symbols_list[t_step][temp_beam_idx_for_trace]
                current_sequence_chars[t_step] = char_symbol.item()
                if char_symbol.item() == eos_id and t_step < self.max_len_labels -1 : # if EOS hit before end
                    # fill rest with EOS, could also break
                     for k_fill in range(t_step + 1, self.max_len_labels): current_sequence_chars[k_fill] = eos_id
                     break
                if t_step > 0:
                    temp_beam_idx_for_trace = stored_predecessors_list[t_step-1][temp_beam_idx_for_trace,0]
            final_sequences.append(current_sequence_chars)
        final_pred_ids = torch.tensor(final_sequences, dtype=torch.long, device=encoder_features_x.device)
        return final_pred_ids, torch.ones_like(final_pred_ids, dtype=torch.float) # Dummy scores




Overwriting /content/aster.pytorch/lib/models/attention_recognition_head.py


In [ ]:
%%writefile /content/aster.pytorch/lib/evaluators.py

from __future__ import print_function, absolute_import
import time
from datetime import datetime
from collections import OrderedDict

import torch
import numpy as np
from random import randint
from PIL import Image
import sys

from . import evaluation_metrics
from .evaluation_metrics.metrics import Accuracy, CharAccuracy, NEDAccuracy, EditDistance, RecPostProcess
from .utils.meters import AverageMeter
from .utils.visualization_utils import recognition_vis, stn_vis
from .utils.labelmaps import labels2strs

metrics_factory = evaluation_metrics.factory()

from config import get_args
#global_args = get_args(sys.argv[1:])

class BaseEvaluator(object):
  def __init__(self, model, metric, use_cuda=True):
    super(BaseEvaluator, self).__init__()
    self.model = model
    self.metric = metric
    self.use_cuda = use_cuda
    self.device = torch.device("cuda" if use_cuda else "cpu")

  def evaluate(self, data_loader, step=1, print_freq=1, tfLogger=None, dataset=None, vis_dir=None):
    self.model.eval()

    batch_time = AverageMeter()
    data_time  = AverageMeter()

    images, outputs, targets, losses = [], {}, [], []
    file_names = []

    end = time.time()
    for i, inputs in enumerate(data_loader):
      data_time.update(time.time() - end)

      input_dict = self._parse_data(inputs)
      output_dict = self._forward(input_dict)

      batch_size = input_dict['images'].size(0)

      total_loss_batch = 0.
      for k, loss in output_dict['losses'].items():
        loss = loss.mean(dim=0, keepdim=True)
        total_loss_batch += loss.item() * batch_size

      images.append(input_dict['images'])
      targets.append(input_dict['rec_targets'])
      losses.append(total_loss_batch)
      if global_args.evaluate_with_lexicon:
        file_names += input_dict['file_name']
      for k, v in output_dict['output'].items():
        if k not in outputs:
          outputs[k] = []
        outputs[k].append(v.cpu())

      batch_time.update(time.time() - end)
      end = time.time()

      if (i + 1) % print_freq == 0:
        print('[{}]\t'
              'Evaluation: [{}/{}]\t'
              'Time {:.3f} ({:.3f})\t'
              'Data {:.3f} ({:.3f})\t'
              .format(datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
                      i + 1, len(data_loader),
                      batch_time.val, batch_time.avg,
                      data_time.val, data_time.avg))

    if not global_args.keep_ratio:
      images = torch.cat(images)
      num_samples = images.size(0)
    else:
      num_samples = sum([subimages.size(0) for subimages in images])
    targets = torch.cat(targets)
    losses = np.sum(losses) / (1.0 * num_samples)
    for k, v in outputs.items():
      outputs[k] = torch.cat(outputs[k])

    if 'pred_rec' in outputs:
      if global_args.evaluate_with_lexicon:
        eval_res = metrics_factory[self.metric+'_with_lexicon'](outputs['pred_rec'], targets, dataset, file_names)
        pred_str = labels2strs(outputs['pred_rec'], dataset.id2char, dataset.char2id)
        print(f"Predictions: {pred_str[:5]}, Targets: {labels2strs(targets, dataset.id2char, dataset.char2id)[:5]}")
        print('lexicon0: {0}, {1:.3f}'.format(self.metric, eval_res[0]))
        print('lexicon50: {0}, {1:.3f}'.format(self.metric, eval_res[1]))
        print('lexicon1k: {0}, {1:.3f}'.format(self.metric, eval_res[2]))
        print('lexiconfull: {0}, {1:.3f}'.format(self.metric, eval_res[3]))
        eval_res = eval_res[0]
      else:
        eval_res = metrics_factory[self.metric](outputs['pred_rec'], targets, dataset)
        char_acc = metrics_factory['CharAccuracy'](outputs['pred_rec'], targets, dataset)
        ned_acc = metrics_factory['NEDAccuracy'](outputs['pred_rec'], targets, dataset)
        ed = metrics_factory['editdistance'](outputs['pred_rec'], targets, dataset)
        pred_str = labels2strs(outputs['pred_rec'], dataset.id2char, dataset.char2id)
        print(f"Predictions: {pred_str[:5]}, Targets: {labels2strs(targets, dataset.id2char, dataset.char2id)[:5]}")
        print('lexicon0: {0}: {1:.3f}'.format(self.metric, eval_res))
        print('lexicon0: CharAccuracy: {0:.3f}'.format(char_acc))
        print('lexicon0: NEDAccuracy: {0:.3f}'.format(ned_acc))
        print('lexicon0: editdistance: {0:.3f}'.format(ed))
      pred_list, targ_list, score_list = RecPostProcess(outputs['pred_rec'], targets, outputs['pred_rec_score'], dataset)

      if tfLogger is not None:
        info = {
          'loss': losses,
          self.metric: eval_res,
          'CharAccuracy': char_acc,
          'NEDAccuracy': ned_acc,
          'editdistance': ed
        }
        for tag, value in info.items():
          tfLogger.scalar_summary(tag, value, step)

    if vis_dir is not None:
      stn_vis(images, outputs['rectified_images'], outputs['ctrl_points'], outputs['pred_rec'],
              targets, score_list, outputs['pred_score'] if 'pred_score' in outputs else None, dataset, vis_dir)
    return eval_res

  def _parse_data(self, inputs):
    input_dict = {}
    if global_args.evaluate_with_lexicon:
      imgs, label_encs, lengths, file_name = inputs
    else:
      imgs, label_encs, lengths = inputs

    with torch.no_grad():
      images = imgs.to(self.device)
      if label_encs is not None:
        labels = label_encs.to(self.device)

    input_dict['images'] = images
    input_dict['rec_targets'] = labels
    input_dict['rec_lengths'] = lengths
    if global_args.evaluate_with_lexicon:
      input_dict['file_name'] = file_name
    return input_dict

  def _forward(self, input_dict):
    self.model.eval()
    with torch.no_grad():
      output_dict = self.model(input_dict)
    return output_dict

class Evaluator(BaseEvaluator):
  def _parse_data(self, inputs):
    input_dict = {}
    if global_args.evaluate_with_lexicon:
      imgs, label_encs, lengths, file_name = inputs
    else:
      imgs, label_encs, lengths = inputs

    with torch.no_grad():
      images = imgs.to(self.device)
      if label_encs is not None:
        labels = label_encs.to(self.device)

    input_dict['images'] = images
    input_dict['rec_targets'] = labels
    input_dict['rec_lengths'] = lengths
    if global_args.evaluate_with_lexicon:
      input_dict['file_name'] = file_name
    return input_dict

  def _forward(self, input_dict):
    self.model.eval()
    with torch.no_grad():
      output_dict = self.model(input_dict)
    return output_dict

Overwriting /content/aster.pytorch/lib/evaluators.py


In [ ]:
import os
import sys
import torch
import torch.nn as nn
import torch.nn.functional as F
# from torch.nn import init # Not directly used in this script block, but ASTER modules use it.

print("--- ASTER Model ONNX Conversion Script (Corrected & Consolidated - with torch.load fix) ---")

# --- Step 1: Define ALL Paths and CRITICAL Configuration ---
# PLEASE VERIFY THESE PATHS CAREFULLY!
ASTER_TRAINED_MODEL_PATH = "/content/drive/MyDrive/pguard/Cars/domainadaptation_thermal_rev3/best.pth.tar"
ASTER_CLONED_REPO_DIR = "/content/aster.pytorch" # Where you cloned ayumiymk/aster.pytorch
ASTER_ONNX_OUTPUT_PATH = "/content/drive/MyDrive/pguard/recognitionmodel.onnx" # Desired output path

# These parameters MUST match how your ASTER model (best.pth.tar) was structured during training.
# These values are based on your provided config.py.
ASTER_MODEL_CONFIG_FOR_EXPORT = {
    'arch': 'ResNet_ASTER',            # From your config.py default
    'voc_type': 'LICENSE_PLATE',       # From your config.py
    'sDim': 512,                       # Corresponds to decoder_sdim in config.py
    'attDim': 512,                     # Corresponds to attDim in config.py
    'max_len': 11,                     # This is 'max_len' from your config.py
    'STN_ON': False,                   # Default from config.py (action='store_true', default=False)
    'with_lstm': True,                 # Default from config.py (action='store_true', default=True)
    # These flags affect ModelBuilder's structure for loading your DA-trained weights:
    'domain_adaptation': True,         # From your config.py (action='store_true', default=True)
    'da_feature_norm': True,           # From your config.py (action='store_true', default=True)
    'label_smoothing': 0.1,            # From your config.py
    # Params for dummy input creation & ONNX wrapper, matching ASTER training:
    'height': 32,                      # From your config.py
    'width': 100,                      # From your config.py
    'beam_width': 5,                   # From your config.py (for AsterONNXWrapper, though .sample() is used for export)
    # STN-specific parameters that ModelBuilder __init__ (from your provided file) expects:
    'tps_inputsize': [32, 64],         # From your config.py
    'tps_outputsize': [32, 100],       # From your config.py
    'num_control_points': 20,          # From your config.py
    'tps_margins': [0.05, 0.05],       # From your config.py
    'stn_activation': 'none',          # From your config.py
    'n_group': 1,                      # Assumed default for ResNet_ASTER if not in config, but your ModelBuilder has it.
}
print(f"  ASTER Trained Model Path: {ASTER_TRAINED_MODEL_PATH}")
print(f"  ASTER Cloned Repo Directory: {ASTER_CLONED_REPO_DIR}")
print(f"  Target ASTER ONNX Output Path: {ASTER_ONNX_OUTPUT_PATH}")
print(f"  Using ASTER Model Config for export: Arch={ASTER_MODEL_CONFIG_FOR_EXPORT['arch']}, STN_ON={ASTER_MODEL_CONFIG_FOR_EXPORT['STN_ON']}")

# --- Step 2: Setup Python Environment for ASTER imports ---
if not os.path.exists(ASTER_TRAINED_MODEL_PATH): print(f"FATAL ERROR: Trained ASTER model file not found: '{ASTER_TRAINED_MODEL_PATH}'"); sys.exit(1)
if not os.path.isdir(ASTER_CLONED_REPO_DIR): print(f"FATAL ERROR: ASTER cloned repository directory not found: '{ASTER_CLONED_REPO_DIR}'"); sys.exit(1)
if ASTER_CLONED_REPO_DIR not in sys.path: sys.path.insert(0, ASTER_CLONED_REPO_DIR)
aster_lib_subpath = os.path.join(ASTER_CLONED_REPO_DIR, "lib")
if aster_lib_subpath not in sys.path: sys.path.insert(0, aster_lib_subpath)
print(f"  Added to sys.path for imports: '{ASTER_CLONED_REPO_DIR}' and '{aster_lib_subpath}'")

# --- Step 3: Import necessary ASTER modules from your CLONED REPOSITORY ---
try:
    from lib.models.model_builder import ModelBuilder
    from lib.utils.labelmaps import get_vocabulary, char2id
    print("  SUCCESS: Imported ModelBuilder and labelmap utilities from the ASTER repo.")
except ImportError as e:
    print(f"  FATAL ERROR: Could not import required ASTER modules. Error: {e}. Check fixes to config.py/resnet_aster.py and runtime restart."); sys.exit(1)
except SystemExit as e:
    print(f"  FATAL ERROR: SystemExit during import (likely argparse in ASTER files). Code: {e.code}. Check fixes and restart runtime."); sys.exit(1)

# --- Step 4: Define the ONNX Wrapper for ASTER's inference path ---
class AsterONNXInferenceWrapper(nn.Module):
    def __init__(self, aster_model_instance_loaded):
        super().__init__(); self.aster_model = aster_model_instance_loaded
    def forward(self, image_batch_tensor):
        current_image_features = image_batch_tensor
        if self.aster_model.STN_ON:
            if hasattr(self.aster_model, '_apply_stn'):
                current_image_features, _ = self.aster_model._apply_stn(image_batch_tensor)
            else: # Fallback if _apply_stn is not a method of your specific ModelBuilder
                 if hasattr(self.aster_model, 'tps_inputsize') and hasattr(self.aster_model, 'stn_head') and hasattr(self.aster_model, 'tps'): # Check if attributes exist
                    if tuple(image_batch_tensor.shape[2:]) != tuple(self.aster_model.tps_inputsize):
                        current_image_features = F.interpolate(image_batch_tensor, self.aster_model.tps_inputsize, mode='bilinear', align_corners=True)
                    _, ctrl_points = self.aster_model.stn_head(current_image_features)
                    current_image_features, _ = self.aster_model.tps(image_batch_tensor, ctrl_points)
                 else:
                    print("WARNING (Wrapper): STN_ON is True but STN components (tps_inputsize, stn_head, tps) not found on model. STN step skipped.")

        encoder_features = self.aster_model.encoder(current_image_features).contiguous()
        predicted_ids_tensor, _ = self.aster_model.decoder.sample(encoder_features)
        return predicted_ids_tensor
print("  AsterONNXInferenceWrapper defined.")

# --- Step 5: Load your trained ASTER model weights and perform ONNX export ---
try:
    print(f"  Instantiating and loading ASTER model for ONNX export...")
    current_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"    Using device: {current_device}")

    cfg = ASTER_MODEL_CONFIG_FOR_EXPORT

    aster_vocab = get_vocabulary(cfg['voc_type'])
    aster_char2id = char2id(aster_vocab)
    eos_id = aster_char2id['EOS']
    rec_num_classes = len(aster_vocab)

    builder_constructor_args = {
        'arch': cfg['arch'], 'rec_num_classes': rec_num_classes, 'sDim': cfg['sDim'],
        'attDim': cfg['attDim'], 'max_len_labels': cfg['max_len'], 'eos': eos_id,
        'STN_ON': cfg['STN_ON'], 'domain_adaptation': cfg['domain_adaptation'],
        'tps_inputsize': tuple(cfg['tps_inputsize']), 'tps_outputsize': tuple(cfg['tps_outputsize']),
        'num_control_points': cfg['num_control_points'], 'tps_margins': tuple(cfg['tps_margins']),
        'stn_activation': cfg['stn_activation'], 'with_lstm': cfg['with_lstm'],
        'n_group': cfg.get('n_group', 1), 'label_smoothing': cfg['label_smoothing'],
        'da_feature_norm': cfg['da_feature_norm']
    }
    print(f"    Final arguments for ModelBuilder instantiation: {builder_constructor_args}")

    aster_model_pytorch = ModelBuilder(**builder_constructor_args)
    print(f"    ASTER ModelBuilder instantiated with arch: {builder_constructor_args['arch']}")

    print(f"    Loading trained weights from: {ASTER_TRAINED_MODEL_PATH}")
    # --- THE CRITICAL FIX IS HERE ---
    checkpoint = torch.load(ASTER_TRAINED_MODEL_PATH, map_location=current_device, weights_only=False)
    # --- END OF CRITICAL FIX ---
    if 'state_dict' not in checkpoint: raise KeyError("'state_dict' key missing from ASTER checkpoint.")

    cleaned_state_dict = {}
    for key, value in checkpoint['state_dict'].items():
        name_to_load = key.replace('module.', ''); skip_param = False
        if not builder_constructor_args['STN_ON'] and (name_to_load.startswith('stn_head.') or name_to_load.startswith('tps.')): skip_param = True
        if 'da_feat_layernorm' in name_to_load and not builder_constructor_args.get('da_feature_norm', False): skip_param = True
        if 'domain_discriminator' in name_to_load and not builder_constructor_args.get('domain_adaptation', False): skip_param = True
        if not skip_param: cleaned_state_dict[name_to_load] = value

    missing_k, unexpected_k = aster_model_pytorch.load_state_dict(cleaned_state_dict, strict=False)
    print(f"    Weights loaded. Missing keys in model: {len(missing_k)}, Unexpected keys from checkpoint: {len(unexpected_k)}.")
    if missing_k and len(missing_k) < 10: print(f"      Sample of missing keys: {missing_k}")
    if unexpected_k and len(unexpected_k) < 10: print(f"      Sample of unexpected keys: {unexpected_k}")

    aster_model_pytorch.eval().to(current_device)
    onnx_wrapper = AsterONNXInferenceWrapper(aster_model_pytorch)
    onnx_wrapper.eval()

    dummy_input = torch.randn(1, 3, ASTER_MODEL_CONFIG_FOR_EXPORT['height'], ASTER_MODEL_CONFIG_FOR_EXPORT['width'], device=current_device)
    print(f"    Dummy input shape for ONNX: {list(dummy_input.shape)}")

    os.makedirs(os.path.dirname(ASTER_ONNX_OUTPUT_PATH), exist_ok=True)
    print(f"    Starting ONNX export to: {ASTER_ONNX_OUTPUT_PATH} (Opset 12)...")
    torch.onnx.export(
        onnx_wrapper, dummy_input, ASTER_ONNX_OUTPUT_PATH,
        input_names=['input_image'], output_names=['output_preds'],
        dynamic_axes={'input_image': {0: 'batch_size'}, 'output_preds': {0: 'batch_size'}},
        opset_version=12, verbose=False
    )
    print(f"    SUCCESS! ASTER model exported to ONNX: {ASTER_ONNX_OUTPUT_PATH}")

except Exception as e:
    print(f"  FATAL ERROR during ASTER ONNX export process: {e}")
    import traceback
    traceback.print_exc()

print("--- ASTER Model ONNX Conversion Script Finished ---")

--- ASTER Model ONNX Conversion Script (Corrected & Consolidated - with torch.load fix) ---
  ASTER Trained Model Path: /content/drive/MyDrive/pguard/Cars/domainadaptation_thermal_rev3/best.pth.tar
  ASTER Cloned Repo Directory: /content/aster.pytorch
  Target ASTER ONNX Output Path: /content/drive/MyDrive/pguard/recognitionmodel.onnx
  Using ASTER Model Config for export: Arch=ResNet_ASTER, STN_ON=False
  Added to sys.path for imports: '/content/aster.pytorch' and '/content/aster.pytorch/lib'
  SUCCESS: Imported ModelBuilder and labelmap utilities from the ASTER repo.
  AsterONNXInferenceWrapper defined.
  Instantiating and loading ASTER model for ONNX export...
    Using device: cpu
    Final arguments for ModelBuilder instantiation: {'arch': 'ResNet_ASTER', 'rec_num_classes': 41, 'sDim': 512, 'attDim': 512, 'max_len_labels': 11, 'eos': 38, 'STN_ON': False, 'domain_adaptation': True, 'tps_inputsize': (32, 64), 'tps_outputsize': (32, 100), 'num_control_points': 20, 'tps_margins': (0.0

/usr/local/lib/python3.11/dist-packages/torch/onnx/symbolic_opset9.py:4277: UserWarning: Exporting a model to ONNX with a batch_size other than 1, with a variable length with LSTM can cause an error when running the ONNX model with a different batch size. Make sure to save the model with a batch size of 1, or define the initial states (h0/c0) as inputs of the model. 
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torch/onnx/symbolic_opset9.py:4277: UserWarning: Exporting a model to ONNX with a batch_size other than 1, with a variable length with GRU can cause an error when running the ONNX model with a different batch size. Make sure to save the model with a batch size of 1, or define the initial states (h0/c0) as inputs of the model. 
  warnings.warn(


    SUCCESS! ASTER model exported to ONNX: /content/drive/MyDrive/pguard/recognitionmodel.onnx
--- ASTER Model ONNX Conversion Script Finished ---


In [ ]:
print("--- Block 6: Defining Inference Pipeline Utilities ---")
# Ensure these imports are available (should be from previous blocks)
import numpy as np
import cv2
from PIL import Image, ImageFile
from torchvision import transforms as T # torchvision.transforms

# If labelmaps was imported as 'from lib.utils.labelmaps import ...' earlier,
# ensure it's accessible or re-import the necessary functions for labels2strs here for clarity.
# Assuming ASTER_CLONED_REPO_DIR is still in sys.path
try:
    from lib.utils.labelmaps import get_vocabulary, char2id, id2char, labels2strs
except ImportError:
    print("  WARNING: Could not re-import ASTER labelmap utilities for Block 6. This might cause issues if not previously imported.")
    # Define a dummy labels2strs if the import fails to prevent immediate crash (though it won't work)
    def labels2strs(ids, id2c, c2id): return [f"LABELDECODE_ERROR_{len(ids)}"]


ImageFile.LOAD_TRUNCATED_IMAGES = True # Good practice

# --- Configuration (from earlier blocks, ensure these are set if running standalone) ---
YOLO_IMG_SIZE_INFER = (640, 640) # W, H
ASTER_IMG_SIZE_INFER = (100, 32) # W, H
ASTER_VOC_TYPE_INFER = 'LICENSE_PLATE'
# --- End Configuration ---

def preprocess_yolo_frame_for_infer(bgr_or_gray_frame, target_size_wh):
    if len(bgr_or_gray_frame.shape) == 2: # Grayscale
        frame_for_yolo = cv2.cvtColor(bgr_or_gray_frame, cv2.COLOR_GRAY2BGR)
    elif bgr_or_gray_frame.shape[2] == 1: # Grayscale with channel dim
        frame_for_yolo = cv2.cvtColor(bgr_or_gray_frame, cv2.COLOR_GRAY2BGR)
    else: # Already 3-channel
        frame_for_yolo = bgr_or_gray_frame
    img_h_o, img_w_o, _ = frame_for_yolo.shape
    target_w, target_h = target_size_wh
    scale = min(target_w / img_w_o, target_h / img_h_o)
    new_w, new_h = int(img_w_o * scale), int(img_h_o * scale)
    img_r = cv2.resize(frame_for_yolo, (new_w, new_h), interpolation=cv2.INTER_LINEAR)
    pad_img = np.full((target_h, target_w, 3), 114, dtype=np.uint8)
    pad_w_off, pad_h_off = (target_w - new_w) // 2, (target_h - new_h) // 2
    pad_img[pad_h_off:pad_h_off+new_h, pad_w_off:pad_w_off+new_w, :] = img_r
    blob = np.transpose(pad_img, (2,0,1)).astype(np.float32) / 255.0
    return blob[np.newaxis, :], scale, pad_w_off, pad_h_off

def postprocess_yolo_onnx_output(yolo_out, conf_thr, iou_thr, scale_val, pad_w, pad_h, original_img_hw):
    preds_raw = yolo_out[0][0]
    if preds_raw.shape[0] == (4 + 1) and preds_raw.shape[1] > 0 : preds_proc = preds_raw.T
    elif preds_raw.shape[1] == (4 + 1) and preds_raw.shape[0] >=0 : preds_proc = preds_raw
    else: print(f"  Warn: YOLO output shape {preds_raw.shape} unexp."); return [],[]
    bxs, cfs, oH, oW = [],[],original_img_hw[0],original_img_hw[1]
    for det_data in preds_proc:
        cx,cy,w_box,h_box,conf = det_data[:5]
        if conf >= conf_thr:
            x1=(cx-w_box/2-pad_w)/scale_val; y1=(cy-h_box/2-pad_h)/scale_val
            x2=(cx+w_box/2-pad_w)/scale_val; y2=(cy+h_box/2-pad_h)/scale_val
            x1c,y1c,x2c,y2c=max(0,int(x1)),max(0,int(y1)),min(oW-1,int(x2)),min(oH-1,int(y2))
            if x2c > x1c and y2c > y1c: bxs.append([x1c,y1c,x2c-x1c,y2c-y1c]); cfs.append(float(conf))
    if not bxs: return [],[]
    ids_nms = cv2.dnn.NMSBoxes(bxs,cfs,conf_thr,iou_thr)
    final_b, final_c = [],[]
    if hasattr(ids_nms,'__len__') and len(ids_nms)>0:
        p_ids = ids_nms[0] if isinstance(ids_nms,tuple) else ids_nms
        for i in p_ids.flatten():final_b.append(bxs[i]);final_c.append(cfs[i])
    return final_b,final_c

aster_pil_transformer_infer = T.Compose([
    T.Resize(ASTER_IMG_SIZE_INFER[::-1], interpolation=T.InterpolationMode.BILINEAR),
    T.ToTensor(), T.Normalize([0.5]*3,[0.5]*3)
])
def preprocess_aster_plate_for_infer(plate_pil_img):
    if plate_pil_img.mode != 'RGB': plate_pil_img = plate_pil_img.convert('RGB')
    img_t = aster_pil_transformer_infer(plate_pil_img)
    return img_t.unsqueeze(0).numpy()

print("--- Inference Utilities Defined ---\n")

--- Block 6: Defining Inference Pipeline Utilities ---
--- Inference Utilities Defined ---



In [ ]:
print("--- Block 7: Executing Video Processing Pipeline (with Self-Contained Corrected labels2strs) ---")

import os
import sys
import time
import cv2
import numpy as np
import onnxruntime
from PIL import Image, ImageFile

ImageFile.LOAD_TRUNCATED_IMAGES = True

# --- Configuration (from Block 1) ---
VIDEO_PATH_IN = "/content/drive/MyDrive/pguard/alpr_thermal.mp4"
YOLO_ONNX_PATH_EXPORTED = "/content/drive/MyDrive/pguard/detection_model.onnx"
ASTER_ONNX_PATH_EXPORTED = "/content/drive/MyDrive/pguard/recognitionmodel.onnx"
VIDEO_PATH_OUT = "/content/alpr_thermal_video_final1.mp4"

YOLO_IMG_SIZE_INFER = (640, 640)
YOLO_CONF_THRES = 0.35
YOLO_IOU_THRES = 0.45
ASTER_VOC_TYPE_INFER = 'LICENSE_PLATE' # For get_vocabulary
ASTER_IMG_SIZE_INFER = (100, 32)    # For ASTER preprocessing transform
# --- End Configuration ---


# --- Self-contained ASTER label utilities FOR THIS SCRIPT ---
# (These are needed if the global import from the ASTER lib isn't reliably updated)
import string # For get_vocabulary

def local_to_numpy(x): # Ensures output is a NumPy array
    if isinstance(x, torch.Tensor): # Check for torch.Tensor if it might be an input
        return x.cpu().detach().numpy() if hasattr(x, 'cpu') else x.detach().numpy()
    elif not isinstance(x, np.ndarray):
        return np.array(x)
    return x

def local_get_vocabulary(voc_type, EOS='EOS', PADDING='PADDING', UNKNOWN='UNKNOWN'):
    voc = None; supported_types = ['LOWERCASE','ALLCASES','ALLCASES_SYMBOLS','LICENSE_PLATE']
    if voc_type == 'LOWERCASE': voc = list(string.digits + string.ascii_lowercase)
    elif voc_type == 'ALLCASES': voc = list(string.digits + string.ascii_letters)
    elif voc_type == 'ALLCASES_SYMBOLS': voc = list(string.printable[:-6])
    elif voc_type == 'LICENSE_PLATE': voc = list('0123456789ABCDEFGHIJKLMNOPQRSTUVWXYZ#-')
    else: raise KeyError(f"LOCAL voc_type '{voc_type}' invalid. Must be one of {supported_types}")
    if EOS not in voc: voc.append(EOS)
    if PADDING not in voc: voc.append(PADDING)
    if UNKNOWN not in voc: voc.append(UNKNOWN)
    return voc

def local_char2id(voc_list): return dict(zip(voc_list, range(len(voc_list))))
def local_id2char(voc_list): return dict(zip(range(len(voc_list)), voc_list))

def local_labels2strs(labels_input, id2char_map_arg, char2id_map_arg):
    np_labels = local_to_numpy(labels_input)
    if np_labels.ndim == 1: np_labels = np.expand_dims(np_labels, 0)
    if np_labels.ndim != 2:
        print(f"WARN (local_labels2strs): Unexpected labels shape {np_labels.shape}.")
        return [f"SHAPE_ERR_{np_labels.shape}" for _ in range(np_labels.shape[0])] if np_labels.shape else ["SHAPE_ERR"]
    strings_list = []; eos_id_val = char2id_map_arg.get('EOS')
    for label_sequence in np_labels:
        current_char_list = []
        for char_code_float_or_int in label_sequence:
            char_code = int(char_code_float_or_int)
            if eos_id_val is not None and char_code == eos_id_val: break
            character = id2char_map_arg.get(char_code)
            if character and character not in ['PADDING', 'UNKNOWN', 'EOS']:
                current_char_list.append(character)
        strings_list.append("".join(current_char_list))
    return strings_list
print("  INFO: Using self-contained ASTER label utilities for Block 7.")
# --- End Self-contained ASTER label utilities ---


# --- Preprocessing/Postprocessing functions (assuming they were defined correctly in Block 6 or here) ---
# Re-define them here to ensure they are in scope for this cell, using the self-contained utilities if needed.
from torchvision import transforms as T # torchvision.transforms
if 'preprocess_yolo_frame_for_infer' not in globals(): # Simple check if defined previously
    def preprocess_yolo_frame_for_infer(bgr_or_gray_frame, target_size_wh):
        if len(bgr_or_gray_frame.shape) == 2: frame_for_yolo = cv2.cvtColor(bgr_or_gray_frame, cv2.COLOR_GRAY2BGR)
        elif bgr_or_gray_frame.shape[2] == 1: frame_for_yolo = cv2.cvtColor(bgr_or_gray_frame, cv2.COLOR_GRAY2BGR)
        else: frame_for_yolo = bgr_or_gray_frame
        img_h_o,img_w_o,_=frame_for_yolo.shape;target_w,target_h=target_size_wh;scale=min(target_w/img_w_o,target_h/img_h_o);new_w,new_h=int(img_w_o*scale),int(img_h_o*scale)
        img_r=cv2.resize(frame_for_yolo,(new_w,new_h),cv2.INTER_LINEAR);pad_img=np.full((target_h,target_w,3),114,np.uint8)
        pad_w_off,pad_h_off=(target_w-new_w)//2,(target_h-new_h)//2;pad_img[pad_h_off:pad_h_off+new_h,pad_w_off:pad_w_off+new_w,:]=img_r
        return (np.transpose(pad_img,(2,0,1)).astype(np.float32)/255.)[np.newaxis,:],scale,pad_w_off,pad_h_off

if 'postprocess_yolo_onnx_output' not in globals():
    def postprocess_yolo_onnx_output(yolo_out,conf_thr,iou_thr,scale_val,pad_w,pad_h,original_img_hw):
        preds_raw=yolo_out[0][0];
        if preds_raw.shape[0]==5 and preds_raw.shape[1]>0:preds_proc=preds_raw.T
        elif preds_raw.shape[1]==5 and preds_raw.shape[0]>=0:preds_proc=preds_raw
        else:print(f"Warn:YOLO out unexp {preds_raw.shape}");return [],[]
        bxs,cfs,oH,oW=[],[],original_img_hw[0],original_img_hw[1]
        for det in preds_proc:
            cx,cy,w_b,h_b,cnf=det[:5]
            if cnf>=conf_thr:
                x1,y1,x2,y2=(cx-w_b/2-pad_w)/scale_val,(cy-h_b/2-pad_h)/scale_val,(cx+w_b/2-pad_w)/scale_val,(cy+h_b/2-pad_h)/scale_val
                x1c,y1c,x2c,y2c=max(0,int(x1)),max(0,int(y1)),min(oW-1,int(x2)),min(oH-1,int(y2))
                if x2c>x1c and y2c>y1c:bxs.append([x1c,y1c,x2c-x1c,y2c-y1c]);cfs.append(float(cnf))
        if not bxs:return [],[]
        ids=cv2.dnn.NMSBoxes(bxs,cfs,conf_thr,iou_thr);fb,fc=[],[]
        if hasattr(ids,'__len__') and len(ids)>0:
            p_ids=ids[0] if isinstance(ids,tuple) else ids
            for i in p_ids.flatten():fb.append(bxs[i]);fc.append(cfs[i])
        return fb,fc

if 'aster_pil_transformer_infer' not in globals():
    aster_pil_transformer_infer=T.Compose([T.Resize(ASTER_IMG_SIZE_INFER[::-1],T.InterpolationMode.BILINEAR),T.ToTensor(),T.Normalize([.5]*3,[.5]*3)])

if 'preprocess_aster_plate_for_infer' not in globals():
    def preprocess_aster_plate_for_infer(pil_img):
        if pil_img.mode!='RGB':pil_img=pil_img.convert('RGB')
        return aster_pil_transformer_infer(pil_img).unsqueeze(0).numpy()
# --- End Preprocessing/Postprocessing functions ---

def run_alpr_video_pipeline(input_video_p, output_video_p, yolo_onnx_p, aster_onnx_p):
    print(f"  Starting ALPR pipeline: Video '{input_video_p}' -> '{output_video_p}'")
    if not all(os.path.exists(p) for p in [input_video_p, yolo_onnx_p, aster_onnx_p]):
        print(f"  ERROR: One or more input files missing. Check paths."); return

    onnx_providers = ['CUDAExecutionProvider', 'CPUExecutionProvider']
    yolo_session, aster_session = None, None
    try:
        yolo_session = onnxruntime.InferenceSession(yolo_onnx_p, providers=onnx_providers)
        aster_session = onnxruntime.InferenceSession(aster_onnx_p, providers=onnx_providers)
    except Exception as e_ort:
        print(f"    Warn: ORT init error ({e_ort}). Fallback to CPU."); onnx_providers = ['CPUExecutionProvider']
        try:
            yolo_session=onnxruntime.InferenceSession(yolo_onnx_p,providers=onnx_providers); aster_session=onnxruntime.InferenceSession(aster_onnx_p,providers=onnx_providers)
        except Exception as e_cpu_ort: print(f"    FATAL: ORT CPU init error: {e_cpu_ort}"); return
    if not yolo_session or not aster_session: print("FATAL: ONNX sessions not created."); return
    print(f"    YOLO ONNX using: {yolo_session.get_providers()}, ASTER ONNX using: {aster_session.get_providers()}")

    # Use the self-contained local_ versions of vocab utilities
    aster_vocab_decode = local_get_vocabulary(ASTER_VOC_TYPE_INFER, EOS='EOS', PADDING='PADDING', UNKNOWN='UNKNOWN')
    aster_id2char_decode = local_id2char(aster_vocab_decode)
    aster_char2id_decode = local_char2id(aster_vocab_decode)

    cap = cv2.VideoCapture(input_video_p);
    if not cap.isOpened(): print(f"ERROR opening video '{input_video_p}'"); return
    vid_w,vid_h,vid_fps = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)),int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)),cap.get(cv2.CAP_PROP_FPS)
    if vid_fps <= 0: vid_fps = 25.0
    if vid_w == 0 or vid_h == 0: print(f"ERROR: Video W/H is zero."); cap.release(); return
    video_writer = cv2.VideoWriter(output_video_p, cv2.VideoWriter_fourcc(*'mp4v'), vid_fps, (vid_w, vid_h))
    print(f"    Output video: {vid_w}x{vid_h} @ {vid_fps:.2f} FPS to '{output_video_p}'")

    frame_num, total_proc_time = 0,0; print("    Processing video frames...")
    while True:
        ret, bgr_frame = cap.read()
        if not ret: break
        orig_crop_src = bgr_frame.copy(); frame_num+=1; loop_st=time.time()

        yolo_blob,scale,pad_w_off,pad_h_off = preprocess_yolo_frame_for_infer(bgr_frame, YOLO_IMG_SIZE_INFER)
        try: yolo_preds = yolo_session.run(None,{yolo_session.get_inputs()[0].name:yolo_blob})
        except Exception as e_yolo: print(f" ERR YOLO fr {frame_num}:{e_yolo}. Skip."); video_writer.write(bgr_frame); continue
        boxes,confs = postprocess_yolo_onnx_output(yolo_preds,YOLO_CONF_THRES,YOLO_IOU_THRES,scale,pad_w_off,pad_h_off,(vid_h,vid_w))

        for idx,(x,y,w,h) in enumerate(boxes):
            x1,y1,x2,y2=max(0,x),max(0,y),min(vid_w,x+w),min(vid_h,y+h)
            if x2<=x1 or y2<=y1: continue
            plate_cv = orig_crop_src[y1:y2,x1:x2]
            if plate_cv.size==0: continue
            plate_pil = Image.fromarray(cv2.cvtColor(plate_cv,cv2.COLOR_BGR2RGB))
            aster_blob = preprocess_aster_plate_for_infer(plate_pil)
            try:
                 aster_ids_out = aster_session.run(None,{aster_session.get_inputs()[0].name:aster_blob})[0]
                 text_aster = local_labels2strs(aster_ids_out, aster_id2char_decode, aster_char2id_decode)[0] # Use local_labels2strs
            except Exception as e_aster: print(f" ERR ASTER fr {frame_num} pl {idx}:{e_aster}."); text_aster="REC_ERR"

            cv2.rectangle(bgr_frame,(x1,y1),(x2,y2),(0,255,0),2)
            label=f"{text_aster}({confs[idx]:.2f})";text_y=y1-10 if y1>25 else y2+20;text_x=max(0,x1);text_y=max(20,min(vid_h-10,text_y))
            cv2.putText(bgr_frame,label,(text_x,text_y),cv2.FONT_HERSHEY_SIMPLEX,0.7,(0,0,255),2)
            if frame_num%(int(vid_fps if vid_fps>1 else 25))==1 and idx==0:print(f"   F{frame_num}:'{text_aster}' Cf:{confs[idx]:.2f}")

        video_writer.write(bgr_frame); total_proc_time+=(time.time()-loop_st)
        if frame_num%(int(vid_fps*5 if vid_fps>1 else 125))==0: print(f"  ...Proc {frame_num} frames. FPS:{(frame_num/total_proc_time if total_proc_time>0 else 0):.2f}")

    avg_fps = frame_num/total_proc_time if total_proc_time > 0 else 0
    print(f"  Done {frame_num} frames. Avg FPS: {avg_fps:.2f}. Saved to: '{output_video_p}'")
    cap.release(); video_writer.release(); cv2.destroyAllWindows()

# --- Execution ---
if not ('YOLO_ONNX_PATH_EXPORTED' in locals() and os.path.exists(YOLO_ONNX_PATH_EXPORTED) and \
      'ASTER_ONNX_PATH_EXPORTED' in locals() and os.path.exists(ASTER_ONNX_PATH_EXPORTED) and \
      'VIDEO_PATH_IN' in locals() and os.path.exists(VIDEO_PATH_IN)):
    print("  ERROR: One or more essential paths (ONNX models, Input Video) are not defined or files missing.")
    print("         Please ensure Block 1 (Setup) and ONNX export blocks (4 & 5) ran successfully.")
else:
    run_alpr_video_pipeline(VIDEO_PATH_IN, VIDEO_PATH_OUT, YOLO_ONNX_PATH_EXPORTED, ASTER_ONNX_PATH_EXPORTED)

print("--- Video Processing Attempt Finished ---\n")

# --- Block 8: Offer Download (Colab Specific - to be run after Block 7) ---
if 'google.colab' in sys.modules and os.path.exists(VIDEO_PATH_OUT):
    from google.colab import files
    try:
        print(f"  Attempting to download '{VIDEO_PATH_OUT}'...")
        files.download(VIDEO_PATH_OUT)
    except Exception as e_colab_dl_final:
        print(f"  Colab download failed for '{VIDEO_PATH_OUT}': {e_colab_dl_final}")
elif os.path.exists(VIDEO_PATH_OUT): # If not Colab, but file exists, inform user
    print(f"  Output video is available at: {os.path.abspath(VIDEO_PATH_OUT)}")
else: # If file doesn't exist (pipeline might have errored before creating it)
    print(f"  Output video '{VIDEO_PATH_OUT}' not found. Skipping download offer.")
print("--- Script Execution Finished ---")

--- Block 7: Executing Video Processing Pipeline (with Self-Contained Corrected labels2strs) ---
  INFO: Using self-contained ASTER label utilities for Block 7.
  Starting ALPR pipeline: Video '/content/drive/MyDrive/pguard/alpr_thermal.mp4' -> '/content/alpr_thermal_video_final1.mp4'
    YOLO ONNX using: ['CUDAExecutionProvider', 'CPUExecutionProvider'], ASTER ONNX using: ['CUDAExecutionProvider', 'CPUExecutionProvider']
    Output video: 1352x1008 @ 30.00 FPS to '/content/alpr_thermal_video_final1.mp4'
    Processing video frames...
   F1:'166TN2161' Cf:0.83
   F31:'166TN2161' Cf:0.83
   F61:'166TN2116' Cf:0.81
   F91:'166TN1158' Cf:0.72
  ...Proc 150 frames. FPS:21.64
  ...Proc 300 frames. FPS:25.57
   F331:'147TN3706' Cf:0.52
   F361:'182TN3055' Cf:0.57
   F391:'101TN8682' Cf:0.69
   F421:'201TN8610' Cf:0.61
  ...Proc 450 frames. FPS:25.27
   F451:'201TN8612' Cf:0.67
   F481:'201TN8610' Cf:0.60
   F511:'201TN8610' Cf:0.66
   F541:'201TN8610' Cf:0.57
   F571:'201TN8610' Cf:0.64
  ..

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

--- Script Execution Finished ---


In [ ]:
!pip install onnxruntime-gpu # Use onnxruntime-gpu for CUDA support, or onnxruntime for CPU only

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 283.2/283.2 MB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 8.9 MB/s eta 0:00:00
